# Task B — smoke test on synthetic data

Purpose: verify the full pipeline (synthetic data → split → baselines → MLP → ranking eval) runs end-to-end in seconds on this laptop. **No real data, no ESM-2, no cluster.** Once this works we know the plumbing is correct and we can swap in real abundance matrices and move heavy training to CSUC.

What we are doing:
- Generate a fake protein-family dataset with a planted signal (~10% of families are 'bioactive' and their abundance is elevated in dysbiotic samples).
- Random split of families into train/val/test.
- Compute 3 baselines: random ranking, ecology score (the simplified MetaWIBELE signal), ElasticNet.
- Train a small MLP on the abundance vectors for 5 epochs.
- Rank test families by predicted P(bioactive), report Precision@K, AUPRC, and enrichment over random.

Success criterion: the MLP and ElasticNet both clearly beat the random baseline. The ecology score sits in between. If that holds, the loss / data / metrics / split code is all correct.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

device: mps


## 1. Synthetic dataset

Mimics the shape of real Task B input:
- `abundance`: matrix of shape `(N_families, N_samples)` — relative abundance of each protein family in each stool sample.
- `is_dysbiotic`: per-sample binary flag, ~40% positive (matches HMP2 roughly).
- `is_bioactive`: per-family binary label, ~10% positive. **This is what we are trying to predict.**

Planted signal: bioactive families have a `+signal_strength` shift in their mean abundance *only in dysbiotic samples*. Non-bioactive families are Gaussian noise. So a model that sees the abundance vector should be able to detect bioactivity, and the simple ecology score (mean abundance in dysbiotic minus mean in non-dysbiotic) should already pick up most of the signal.

In [2]:
def make_synthetic_taskb(n_families=500, n_samples=200, frac_positive=0.10,
                         frac_dysbiotic=0.40, signal_strength=0.9,
                         confounder_frac=0.20, seed=0):
    """
    Planted signal: bioactive families have a small mean shift in dysbiotic samples.
    Realistic noise: `confounder_frac` of NON-bioactive families also have a shift
    (so ecology score alone is not a perfect predictor) and every family has a
    random per-family noise variance.
    """
    rng = np.random.default_rng(seed)
    is_dysbiotic = rng.random(n_samples) < frac_dysbiotic
    is_bioactive = np.zeros(n_families, dtype=int)
    n_pos = int(round(n_families * frac_positive))
    is_bioactive[:n_pos] = 1
    rng.shuffle(is_bioactive)
    per_fam_noise = rng.uniform(0.5, 2.0, size=n_families).astype(np.float32)
    abundance = (rng.standard_normal((n_families, n_samples)).astype(np.float32)
                 * per_fam_noise[:, None])
    for i in np.where(is_bioactive == 1)[0]:
        abundance[i, is_dysbiotic] += signal_strength * rng.uniform(0.5, 1.5)
    neg_idx = np.where(is_bioactive == 0)[0]
    n_conf = int(round(len(neg_idx) * confounder_frac))
    for i in rng.choice(neg_idx, size=n_conf, replace=False):
        abundance[i, is_dysbiotic] += signal_strength * rng.uniform(0.3, 1.0)
    return abundance, is_dysbiotic, is_bioactive

abundance, is_dysbiotic, is_bioactive = make_synthetic_taskb(seed=SEED)
print(f'abundance:   {abundance.shape}  (families × samples)')
print(f'dysbiotic:   {is_dysbiotic.sum():4d} / {len(is_dysbiotic):4d} samples')
print(f'bioactive:   {is_bioactive.sum():4d} / {len(is_bioactive):4d} families  ({is_bioactive.mean()*100:.1f}%)')

abundance:   (500, 200)  (families × samples)
dysbiotic:     70 /  200 samples
bioactive:     50 /  500 families  (10.0%)


## 2. Train / val / test split

Random family-level split. Stratified by `is_bioactive` so each fold has the same positive fraction. (When we move to real data we will revisit whether nearly identical families need to be grouped to prevent sequence leakage.)

In [3]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(is_bioactive))
idx_trval, idx_te = train_test_split(idx, test_size=0.20, stratify=is_bioactive, random_state=SEED)
idx_tr, idx_va    = train_test_split(idx_trval, test_size=0.20,
                                     stratify=is_bioactive[idx_trval], random_state=SEED)
for name, ix in [('train', idx_tr), ('val', idx_va), ('test', idx_te)]:
    print(f'{name:5s}: {len(ix):4d} families  ({is_bioactive[ix].sum():3d} positive, '
          f'{is_bioactive[ix].mean()*100:.1f}%)')

train:  320 families  ( 32 positive, 10.0%)
val  :   80 families  (  8 positive, 10.0%)
test :  100 families  ( 10 positive, 10.0%)


## 3. Ranking metrics

Same shape as the metrics we will use against MetaWIBELE on real data.

In [4]:
def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    top = np.argsort(-scores)[:k]
    return float(y_true[top].mean())

def enrichment_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    """Precision@k divided by base rate. 1.0 = random, >1 = better than random."""
    base = float(y_true.mean())
    if base == 0: return float('nan')
    return precision_at_k(y_true, scores, k) / base

def report(name: str, y_true: np.ndarray, scores: np.ndarray, results: list, ks=(10, 50)):
    row = {'method': name, 'auprc': average_precision_score(y_true, scores),
           'auroc': roc_auc_score(y_true, scores)}
    for k in ks:
        row[f'P@{k}']   = precision_at_k(y_true, scores, k)
        row[f'enr@{k}'] = enrichment_at_k(y_true, scores, k)
    results.append(row)
    print(f"{name:30s}  AUPRC={row['auprc']:.3f}  AUROC={row['auroc']:.3f}  "
          f"P@10={row['P@10']:.2f} (enr={row['enr@10']:.1f}x)  "
          f"P@50={row['P@50']:.2f} (enr={row['enr@50']:.1f}x)")

results = []

## 4. Baselines

Three baselines on the test set, in order of expected difficulty:
1. **Random** — floor.
2. **Ecology score** — `mean(abundance | dysbiotic) - mean(abundance | non_dysbiotic)`. Simplified MetaWIBELE signal. Should easily detect the planted shift.
3. **ElasticNet** — logistic regression with L1+L2 penalty fit on standardized abundance vectors. The strong classical baseline from Task A.

In [5]:
rng = np.random.default_rng(SEED)

rand_scores = rng.random(len(idx_te))
report('Random', is_bioactive[idx_te], rand_scores, results)

eco_scores_all = abundance[:, is_dysbiotic].mean(axis=1) - abundance[:, ~is_dysbiotic].mean(axis=1)
report('Ecology score', is_bioactive[idx_te], eco_scores_all[idx_te], results)

scaler = StandardScaler().fit(abundance[idx_tr])
X_tr_s = scaler.transform(abundance[idx_tr])
X_te_s = scaler.transform(abundance[idx_te])
enet = LogisticRegression(solver='saga', l1_ratio=0.5, C=1.0,
                          max_iter=2000, class_weight='balanced',
                          random_state=SEED)
enet.fit(X_tr_s, is_bioactive[idx_tr])
enet_scores = enet.predict_proba(X_te_s)[:, 1]
report('ElasticNet', is_bioactive[idx_te], enet_scores, results)

Random                          AUPRC=0.120  AUROC=0.539  P@10=0.10 (enr=1.0x)  P@50=0.12 (enr=1.2x)
Ecology score                   AUPRC=0.799  AUROC=0.958  P@10=0.80 (enr=8.0x)  P@50=0.20 (enr=2.0x)


ElasticNet                      AUPRC=0.490  AUROC=0.878  P@10=0.50 (enr=5.0x)  P@50=0.18 (enr=1.8x)


## 5. Small MLP ranker

Two hidden layers, ReLU, dropout, BCE loss with positive-class weight. Trained for 5 epochs only — this is a smoke test, not a final model.

In [6]:
class AbundanceRanker(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 64, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_ranker(X_tr, y_tr, X_va, y_va, epochs=5, batch_size=64, lr=1e-3):
    Xtr = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
    Xva = torch.tensor(X_va, dtype=torch.float32, device=DEVICE)
    yva = torch.tensor(y_va, dtype=torch.float32, device=DEVICE)
    n_pos = float(y_tr.sum()); n_neg = float(len(y_tr) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], device=DEVICE)
    print(f'  pos_weight = {pos_weight.item():.2f}  (n_pos={int(n_pos)}, n_neg={int(n_neg)})')
    model = AbundanceRanker(Xtr.shape[1]).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va_logits = model(Xva).cpu().numpy()
            va_auprc = average_precision_score(y_va, va_logits)
        print(f'  epoch {ep+1}/{epochs}  val AUPRC = {va_auprc:.3f}')
    return model

mlp = train_ranker(X_tr_s, is_bioactive[idx_tr].astype(np.float32),
                   scaler.transform(abundance[idx_va]), is_bioactive[idx_va].astype(np.float32),
                   epochs=5)

with torch.no_grad():
    mlp_scores = mlp(torch.tensor(X_te_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
report('MLP (5 epochs)', is_bioactive[idx_te], mlp_scores, results)

  pos_weight = 9.00  (n_pos=32, n_neg=288)


  epoch 1/5  val AUPRC = 0.302
  epoch 2/5  val AUPRC = 0.438
  epoch 3/5  val AUPRC = 0.507
  epoch 4/5  val AUPRC = 0.588
  epoch 5/5  val AUPRC = 0.649
MLP (5 epochs)                  AUPRC=0.638  AUROC=0.919  P@10=0.60 (enr=6.0x)  P@50=0.20 (enr=2.0x)


## 6. Summary

In [7]:
res = pd.DataFrame(results).set_index('method').round(3)
print(res.to_string())
print()
best = res['auprc'].idxmax()
print(f'best AUPRC on test: {best} ({res.loc[best, "auprc"]:.3f})')
base_rate = float(is_bioactive[idx_te].mean())
assert res.loc['Random', 'auprc'] < base_rate + 0.10, \
    'Random baseline AUPRC unexpectedly high — check label generation'
assert res.loc['MLP (5 epochs)', 'auprc'] > base_rate + 0.10, \
    'MLP did not beat random by a margin — pipeline is broken'
assert res.loc['ElasticNet', 'auprc'] > base_rate + 0.10, \
    'ElasticNet did not beat random — pipeline is broken'
assert res.loc['Ecology score', 'auprc'] > base_rate + 0.10, \
    'Ecology score did not beat random — synthetic signal is too weak'
print('SMOKE TEST PASSED')

                auprc  auroc  P@10  enr@10  P@50  enr@50
method                                                  
Random          0.120  0.539   0.1     1.0  0.12     1.2
Ecology score   0.799  0.958   0.8     8.0  0.20     2.0
ElasticNet      0.490  0.878   0.5     5.0  0.18     1.8
MLP (5 epochs)  0.638  0.919   0.6     6.0  0.20     2.0

best AUPRC on test: Ecology score (0.799)
SMOKE TEST PASSED


## 7. What this proves and what's next

Proves:
- Data loading, split, training loop, ranking metrics all work end-to-end.
- The MLP can learn from abundance vectors and produces a usable ranking.
- Baselines behave as expected: random ≈ base rate; ecology + ElasticNet detect the planted signal.

Does not prove anything about the real task. Real data will have:
- Much weaker signal (real bioactive proteins do not have a clean +1.5 shift).
- Many more features (~1,638 samples per family) and many more families (~1M).
- Imbalanced labels with fewer than 10% positives.
- Sequence information that this model ignores.

Next:
1. `01_data.ipynb` — download MetaWIBELE outputs from `ibdmdb.org/results` and the supplementary tables (positive labels) from Zhang 2022.
2. `02_abundance_only.ipynb` — re-run this notebook with real abundance + real labels, no synthetic data. Same model, same metrics.
3. `03_sequence.ipynb` — on CSUC cluster: compute ESM-2 embeddings once per family, save to disk, train fused model.

## Double ML — deliberately skipped

Double/Debiased ML estimates the causal effect of one variable on another while controlling for many confounders. It is the right tool when the question is *causal*: e.g. "after controlling for parent species abundance, does this protein's own enrichment in dysbiotic samples predict bioactivity?". Task B as framed is *predictive ranking* — we want the highest possible Precision@K on a held-out set, just like MetaWIBELE. Same flavor of task on both sides, so DML is unnecessary and would make the comparison harder to interpret. Revisit only if the supervisor asks for a causal claim about a specific protein.